In [ ]:
import matplotlib.pyplot as plt
from pandas import DataFrame
from common.utils import load_dataset, optimize_memory, get_params, DatasetType
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from plotnine import ggplot, aes, geom_histogram, scale_x_log10, scale_y_sqrt
import pandas as pd

In [ ]:
df: DataFrame = load_dataset("nyc-taxi-trip-duration", DatasetType.TRAIN, index=True)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

#### Missing values

In [ ]:
missing_counts = df.isna().sum()
missing_counts.plot(kind='bar')
plt.ylabel('Number of Missing Values')
plt.title('Missing Values per Column')
plt.show()

In [ ]:
df, na_list = optimize_memory(df, deep=True)
df.dtypes

In [ ]:
df.head()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])
df.head()

In [ ]:
df.dtypes

##### Validate the trip duration with the pickup and drop off time

In [ ]:
df['check'] = ((df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds()
               + df['trip_duration']).abs() > 0

result = df[['check', 'pickup_datetime', 'dropoff_datetime', 'trip_duration']] \
    .groupby('check') \
    .size() \
    .reset_index(name='n')

print(result)

##### Trip duration plot

In [ ]:
(
    ggplot(df, aes(x='trip_duration'))
    + geom_histogram(fill='red', bins=150)
    + scale_x_log10()
    + scale_y_sqrt()
)

In [ ]:
df_sorted = df.sort_values('trip_duration', ascending=False)

cols = ['trip_duration', 'pickup_datetime', 'dropoff_datetime'] + \
       [col for col in df.columns if col not in ['trip_duration', 'pickup_datetime', 'dropoff_datetime']]
df_sorted = df_sorted[cols]
df_sorted.head(10)

##### Trip duration by month

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=False)

axes[0].hist(df['pickup_datetime'], bins=120, color='red')
axes[0].set_xlabel('Pickup dates')
axes[0].set_ylabel('Count')

axes[1].hist(df['dropoff_datetime'], bins=120, color='blue')
axes[1].set_xlabel('Dropoff dates')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()